# 00 · 环境配置与项目导览

欢迎来到 **深度学习手写教学项目**！

这个项目的目标只有一个：**带你从零亲手实现深度学习的核心，而不是只学会"调包"。**

我们会从最朴素的"用梯度下降拟合一条直线"开始，一步步造出一个 API 贴近 PyTorch 的微型框架 `minitorch`，并最终用它训练 CNN、RNN/LSTM 和 Transformer。

> 📌 本 notebook 不需要你懂任何深度学习知识，只要会基本的 Python。我们先把环境配好、把"地图"看清楚。

## 学习目标

- ✅ 配好并**自检**运行环境（NumPy / Matplotlib / 可选的 PyTorch）
- ✅ 安装并导入我们将一起构建的 `minitorch` 包
- ✅ 认识贯穿全程的核心验证工具：**数值梯度检查 `gradcheck`**
- ✅ 看懂整个项目的**学习路线图**与每个 notebook 的**统一结构**

## 全景路线图：我们要造什么

整个项目是一条"造轮子"的主线——每一步都建立在上一步之上：

```
手推梯度 → 标量自动求导引擎(Value) → 张量自动求导(Tensor) →
nn.Module / 层 → 优化器 / 数据加载 → CNN → RNN/LSTM → Transformer
```

| Part | 主题 | 你将亲手实现 |
|:--:|---|---|
| 1 | 数学与梯度直觉 | 线性回归、梯度下降、手推反向传播 |
| 2 | 标量自动求导 | `Value` 引擎（micrograd 风格） |
| 3 | 张量自动求导 | `Tensor` + 广播 / matmul 反向（**框架心脏**）|
| 4 | 神经网络抽象 | `Module` / `Linear` / 激活 / 损失，训练 MLP 跑 MNIST |
| 5 | 优化与训练工程 | SGD / Adam、DataLoader、Dropout、BatchNorm / LayerNorm |
| 6 | 卷积神经网络 | `Conv2d`(im2col)、池化，训练 CNN |
| 7 | 循环神经网络 | `RNNCell` / `LSTMCell`、随时间反向传播 BPTT |
| 8 | 注意力与 Transformer | 注意力、多头、位置编码，训练 mini-Transformer |

## 每个 notebook 的统一结构

为方便学习，从 Part 1 起，每个核心 notebook 都遵循相同的 **7 段式结构**：

1. **学习目标** —— 这一节读完你能做什么
2. **直觉与数学原理** —— 先讲人话，再给必要公式
3. **从零手写实现** —— 只用 NumPy，一步步实现
4. **验证** —— 用 `gradcheck` 数值检查，或与解析解 / PyTorch 对齐，打印 ✅/❌
5. **PyTorch 对照** —— 同样的功能在 PyTorch 里怎么写，确认结果一致
6. **沉淀进 minitorch** —— 把定稿代码收进包，后续直接复用
7. **小练习** —— 2~3 道动手题，巩固理解

> 💡 **建议**：按编号顺序学习；每个 notebook 都可独立分段运行，但请先完成下面的「环境自检」并执行过一次 `pip install -e .`。

## 1. 环境自检

先确认基础库就位。下面几格逐一检查 Python、NumPy、Matplotlib 与（可选的）PyTorch。

In [ ]:
import sys, platform
print("Python    :", sys.version.split()[0], "  on", platform.platform())

import numpy as np
print("NumPy     :", np.__version__)

import matplotlib
print("Matplotlib:", matplotlib.__version__)

In [ ]:
# torch / torchvision 仅用于后续「PyTorch 对照」章节，这里检查是否已安装
try:
    import torch, torchvision
    print("PyTorch    :", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("CUDA 可用   :", torch.cuda.is_available(), " （本项目用 CPU 即可）")
except ImportError:
    print("⚠️  未检测到 torch / torchvision —— 不影响手写部分，但「对照」章节需要它。")
    print("    安装： pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu")

## 2. 安装并导入 minitorch

如果你还没装过本项目的包，请在**项目根目录**执行一次：

```bash
pip install -e .
```

它会以"可编辑模式"安装 `minitorch`——这样课程中对包的任何修改都会立即生效，无需重装。

In [ ]:
import minitorch
from minitorch import set_seed, gradcheck, numerical_gradient

print("minitorch 版本:", minitorch.__version__)
set_seed(42)
print("已设定随机种子 = 42（保证结果可复现）")

## 3. 核心武器：数值梯度检查

反向传播的梯度公式很容易写错。幸运的是，我们有一个几乎不会错的"真值"可以对照——**数值梯度**：

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + \epsilon\, e_i) - f(x - \epsilon\, e_i)}{2\epsilon}$$

只要把每个输入分量轻微地"加一点、减一点"，看函数值怎么变，就能估计出导数。我们用它来"验收"自己手写的每一个梯度公式。这个工具就是 `minitorch.gradcheck`，本项目会反复用到它。

下面做个最小演示：函数 $f(x)=\sum_i x_i^2$ 的解析梯度是 $2x$，我们让 `gradcheck` 来验证它。

In [ ]:
import numpy as np

x = np.random.randn(4)
f = lambda v: np.sum(v ** 2)     # 标量函数
analytic = 2 * x                 # 我们"手写"的解析梯度

gradcheck(f, x, analytic, name="f(x)=sum(x^2)")

print("数值梯度 :", np.round(numerical_gradient(f, x), 4))
print("解析梯度 :", np.round(analytic, 4))

## 4. 可复现性 + 画图自检

最后确认两件小事：随机种子能让结果可复现；Matplotlib 能正常出图。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

set_seed(7); a = np.random.randn(5)
set_seed(7); b = np.random.randn(5)
print("两次设同一种子，结果一致:", np.allclose(a, b))

xs = np.linspace(-5, 5, 200)
plt.figure(figsize=(5.5, 3))
plt.plot(xs, 1 / (1 + np.exp(-xs)), label="sigmoid")
plt.plot(xs, np.tanh(xs), label="tanh")
plt.plot(xs, np.maximum(0, xs), label="relu")
plt.title("几个常见激活函数（先混个眼熟）")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

## 关于数据

- **图像任务（Part 4 / 6）**：使用经典的 **MNIST** 手写数字数据集，通过 `torchvision` 自动下载并缓存到 `data/`（只下载一次）。
- **序列任务（Part 7 / 8）**：一律使用**合成数据**（如序列复制、字符级加法），无需任何下载，CPU 上秒级到分钟级即可训练。
- 所有"真训练"都刻意控制规模，确保在**纯 CPU、几分钟内**跑完——我们追求的是**理解原理**，而非刷榜性能。

## 小结 & 下一站

如果上面的格子都顺利运行（尤其 `gradcheck` 打印了 ✅），说明环境已就绪 🎉。

**下一站 → `part1_foundations/01_what_is_learning.ipynb`**：我们将用最简单的线性回归，亲手实现"模型 / 损失 / 梯度下降"这套贯穿所有深度学习的核心循环。